# recs_020 — V2a-embed taxonomy USE cosine spike

**Semantic metadata rerank on frozen `two_tower_v1` @100 pools · train_tune → val**

Follow-up to V2a Jaccard (`recs_019`): use **USE embeddings on resolved taxonomy tag names** so related tags (e.g. Action vs Thriller) can score above zero without exact FK overlap.

**Plan:** [`docs/recommender_v2_plan.md`](../../docs/recommender_v2_plan.md) · **Jaccard spike:** [`recs_019_v2a_metadata_jaccard.ipynb`](recs_019_v2a_metadata_jaccard.ipynb) · **Doc:** [`docs/implementations/v2a_metadata_jaccard.md`](../../docs/implementations/v2a_metadata_jaccard.md)


# Executive Summary

**Question:**  
Does taxonomy USE cosine reranking beat D1 — including **D1 + embed metadata** (`*_embed_query_logpop_blend`)?

**Result:**  
Pure retr+meta embed (**`two_tower_v1_v2a_embed_query`**) fails vs D1: **0.052** / **0.035** overall / Slice A. **`two_tower_v1_v2a_embed_query_logpop_blend`** (D1 + pooled taxonomy USE, `w_meta=0.1`, `genre_theme_kw`) **beats D1** on val: overall NDCG@10 **0.095** / Slice A **0.070** vs D1 **0.093** / **0.068**; personalization gap **0.726** vs D1 **0.720** (guardrail passes). Best val Slice A among v2 spikes run so far.

**Recommendation / Decision:**  
**Kill** retr+meta-only embed for ship. **Promotion candidate:** `two_tower_v1_v2a_embed_query_logpop_blend`. Head-to-head vs `recs_019` Jaccard logpop_blend, then wire winner into eval job + decision log.


# Business Context

V2a Jaccard improved train_tune but **did not beat D1 on val**. Exact FK overlap misses related tags. Job 2 already materializes taxonomy USE vectors — this spike tests whether **semantic tag similarity** fixes that gap on the same frozen pools.


# Research Question

On `train_ranker_v1` / `val_dev_12k_v1`, does weighted multi-field **cosine similarity** on taxonomy USE embeddings rerank frozen pools better than Jaccard, D1, and bare `two_tower_v1`?


# Hypothesis

Embedding resolved tag **names** (same USE as v1) captures near-sibling concepts Jaccard cannot. **Pooled** field vectors are the default; **entity_max** (max cosine over per-tag vectors) is an ablation.

**Success criteria:** Beat D1 on val Slice A NDCG@10 without worsening personalization vs D1.


# Definitions

| Term | Definition |
|------|------------|
| V2a-embed | Taxonomy USE cosine rerank (this spike) |
| `{field}_names__use_pooled` | One L2-normalized 512-d vector per field per game (mean of tag embeddings) |
| `{field}_names__use` | One 512-d vector per resolved tag (variable length per game) |
| **pooled** sim | Cosine between anchor and candidate pooled field vectors |
| **entity_max** sim | Per field: max cosine between any anchor tag vector and any candidate tag vector |
| α | Retrieval blend weight (same as V2a Jaccard spike) |


# Data Sources

| Source | Path |
|--------|------|
| Train pools | `artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet` |
| Val pools | `artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl` |
| Val examples (history) | `artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet` |
| IGDB enriched (USE cols) | `artifacts/igdb/igdb_games__enriched.parquet` |

Prerequisites: same as `recs_019` (IGDB Job 1 + Job 2, train pools export, offline eval jsonl).


# Design / Process

1. Load pooled + entity USE columns per taxonomy field from enriched parquet
2. Score pool candidates: weighted mean of field cosines vs query/history anchor
3. Grid-search **sim_mode** (`pooled` / `entity_max`) × field preset × α on **train_tune** (query anchor)
4. Val face-off vs baselines + oracle; full ranking metric tables

```bash
python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json
python scripts/recs_job_igdb_games_enriched.py configs/recs_job_igdb_games_enriched.json
python scripts/recs_job_export_retrieval_pools.py configs/recs_job_export_retrieval_pools_train_ranker.json
python scripts/recs_job_eval_offline.py configs/recs_job_eval_offline.json \
  --examples-parquet artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
```


# Decision Log

| Decision | Reason |
|----------|--------|
| Pooled as default sim_mode | Cheaper; one vector per field; stable anchor for history (mean then L2) |
| entity_max ablation | Tests whether per-tag matching beats pooled field summary |
| Exclude summary/storyline USE | V2b scope — this spike is taxonomy-only |
| Tune query anchor on train | `train_ranker_v1` has no train_review_rows |


# Evaluation Outputs / Artifacts

| Artifact | Path |
|----------|------|
| Train grid | `artifacts/recs/spikes/v2a_embed/v2a_embed_train_tune_grid.csv` |
| Val per-example | `artifacts/recs/spikes/v2a_embed/v2a_embed_val_per_example.parquet` |
| Val overall / slice / personalization | `artifacts/recs/spikes/v2a_embed/v2a_embed_val_*.csv` |
| `v2a_embed_plus_d1_train_tune_grid.csv` | `artifacts/recs/spikes/v2a_embed/` | D1+meta `w_meta` grid |


# Notebook Roadmap

1. Setup and paths
2. Load pools, catalog, IGDB USE embeddings, val history
3. Validate embedding coverage
4. Cosine scoring helpers + train_tune grid
5. Val face-off + metric tables
6. Findings


# Analysis

## Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable, Literal

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import load_retrieval_pool_rows, load_retrieval_pools_jsonl
from steam_review_ml.evaluation.heuristic_ranker import (
    DEFAULT_LOGPOP_BLEND_ALPHA,
    minmax_norm,
    score_logpop_blend,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    RANKING_REPORT_METRIC_COLS,
    _append_personalization_metrics,
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    _table_by_slice_for_metrics,
    _table_by_support_for_metrics,
    _table_overall_ranking,
    _table_personalization,
    _table_popularity,
    average_precision_at_k,
    hit_rate_at_k,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
)
from steam_review_ml.igdb.constants import TAXONOMY_RESOLVE_FIELDS
from steam_review_ml.recommender.retrieve import ContentRetriever

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
SPLIT_SEED = 42
TUNE_FRAC = 0.10
K_FINAL = 10
K_PERSONALIZATION = 10
MIN_REVIEW_CHARS = 30
POOL_METHOD = "two_tower_v1"
D1_ALPHA = DEFAULT_LOGPOP_BLEND_ALPHA

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
VAL_EXAMPLES_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"
IGDB_ENRICHED = REPO_ROOT / "artifacts/igdb/igdb_games__enriched.parquet"
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
SPIKE_OUT = ARTIFACT_DIR / "spikes/v2a_embed"

TAXONOMY_FIELDS: tuple[str, ...] = TAXONOMY_RESOLVE_FIELDS
POOLED_COL = {f: f"{f}_names__use_pooled" for f in TAXONOMY_FIELDS}
ENTITY_COL = {f: f"{f}_names__use" for f in TAXONOMY_FIELDS}

FIELD_PRESETS: dict[str, tuple[str, ...]] = {
    "all5": TAXONOMY_FIELDS,
    "no_keywords": ("genres", "themes", "game_modes", "player_perspectives"),
    "genre_theme": ("genres", "themes"),
    "genre_theme_kw": ("genres", "themes", "keywords"),
}

SIM_MODES: tuple[Literal["pooled", "entity_max"], ...] = ("pooled", "entity_max")
BLEND_ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
PLUS_D1_WEIGHTS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3]  # w on metadata; 0 = D1 only

for p in (TRAIN_POOLS_PARQUET, VAL_JSONL, VAL_EXAMPLES_PARQUET, IGDB_ENRICHED):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

SPIKE_OUT.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT={REPO_ROOT}")
print(f"SPIKE_OUT={SPIKE_OUT}")


/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-22 14:51:55.170111: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-22 14:51:55.180627: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782154315.193341 3543218 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782154315.197110 3543218 cuda_bl

REPO_ROOT=/home/ryanr/workspace/steam_recommendations
SPIKE_OUT=/home/ryanr/workspace/steam_recommendations/artifacts/recs/spikes/v2a_embed


## Load Data

In [ ]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)
val_pools_by_ex = {int(r["ex_idx"]): r for r in val_pools}

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT, min_review_chars=MIN_REVIEW_CHARS, artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

retriever = ContentRetriever(artifact_dir=ARTIFACT_DIR, repo_root=REPO_ROOT)
X_emb = retriever.embedding_matrix

val_examples_df = pd.read_parquet(VAL_EXAMPLES_PARQUET)
history_apps_by_ex: dict[int, set[int]] = {}
examples_for_pers: list[dict[str, Any]] = []
for _, row in val_examples_df.iterrows():
    ex_idx = int(row["ex_idx"])
    train_rows = json.loads(row["train_review_rows_json"])
    history_apps_by_ex[ex_idx] = {int(r["app_id"]) for r in train_rows}
    examples_for_pers.append(
        {
            "ex_idx": ex_idx,
            "user_id": row["user_id"],
            "query_app_id": int(row["query_app_id"]),
            "query_ts": float(row["query_ts"]),
            "n_eval_targets": int(row["n_eval_targets"]),
            "train_review_rows": train_rows,
            "validation_positive_app_ids": json.loads(row["validation_positive_app_ids_json"]),
        }
    )

support_by_ex = val_examples_df.set_index("ex_idx")["n_support_train"].astype(int).to_dict()
for row in val_pools:
    row["n_support_train"] = int(support_by_ex.get(int(row["ex_idx"]), 0))

print(
    f"train={len(train_pools):,} val={len(val_pools):,} catalog={len(app_ids):,} "
    f"val_w_history={sum(1 for s in history_apps_by_ex.values() if s):,}"
)


train=51,691 val=12,500 catalog=315 val_w_history=9,375


## Validate Data Quality

In [3]:
def parse_entity_use(val: Any) -> np.ndarray:
    """Normalize ``{field}_names__use`` to shape (n_tags, 512).

    Parquet stores a 1-d object array of per-tag vectors, not a flat float matrix.
    """
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.zeros((0, 512), dtype=np.float64)
    if isinstance(val, np.ndarray):
        if val.ndim == 2:
            return np.asarray(val, dtype=np.float64)
        if val.ndim == 1 and val.dtype != object and val.size == 512:
            return np.asarray(val, dtype=np.float64).reshape(1, 512)
        if val.ndim == 1:
            if len(val) == 0:
                return np.zeros((0, 512), dtype=np.float64)
            return np.stack([np.asarray(x, dtype=np.float64) for x in val], axis=0)
    if isinstance(val, (list, tuple)):
        if not val:
            return np.zeros((0, 512), dtype=np.float64)
        return np.stack([np.asarray(x, dtype=np.float64) for x in val], axis=0)
    return np.zeros((0, 512), dtype=np.float64)


def parse_pooled_use(val: Any) -> np.ndarray:
    """Normalize ``{field}_names__use_pooled`` to a 512-d vector (zeros if missing/empty)."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.zeros(512, dtype=np.float64)
    arr = np.asarray(val, dtype=np.float64)
    if arr.size == 512:
        return arr.reshape(512)
    return np.zeros(512, dtype=np.float64)


use_cols = [POOLED_COL[f] for f in TAXONOMY_FIELDS] + [ENTITY_COL[f] for f in TAXONOMY_FIELDS]
igdb_df = pd.read_parquet(IGDB_ENRICHED, columns=["app_id", *use_cols])

pooled_by_app: dict[int, dict[str, np.ndarray]] = {}
entity_by_app: dict[int, dict[str, np.ndarray]] = {}

for _, row in igdb_df.iterrows():
    app_id = int(row["app_id"])
    pooled_by_app[app_id] = {}
    entity_by_app[app_id] = {}
    for f in TAXONOMY_FIELDS:
        pooled_by_app[app_id][f] = parse_pooled_use(row[POOLED_COL[f]])
        entity_by_app[app_id][f] = parse_entity_use(row[ENTITY_COL[f]])

catalog_app_set = {int(a) for a in app_ids}
nonzero_pooled = {
    f: sum(1 for a in catalog_app_set if np.linalg.norm(pooled_by_app.get(a, {}).get(f, 0)) > 1e-8)
    for f in TAXONOMY_FIELDS
}
display(pd.Series(nonzero_pooled, name="catalog_apps_with_pooled_emb"))


genres                 315
themes                 310
keywords               287
game_modes             315
player_perspectives    302
Name: catalog_apps_with_pooled_emb, dtype: int64

## Feature Engineering / Preprocessing

In [4]:
AnchorMode = Literal["query", "history"]
SimMode = Literal["pooled", "entity_max"]


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity; returns 0.0 if either vector has zero norm."""
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    if na <= 1e-12 or nb <= 1e-12:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def mean_l2_normalize(vecs: list[np.ndarray]) -> np.ndarray:
    """Mean of vectors then L2-normalize (history anchor for pooled mode)."""
    valid = [np.asarray(v, dtype=np.float64).reshape(512) for v in vecs if np.asarray(v).size == 512]
    if not valid:
        return np.zeros(512, dtype=np.float64)
    if len(valid) == 1:
        v = valid[0]
        n = float(np.linalg.norm(v))
        return v / n if n > 1e-12 else np.zeros(512, dtype=np.float64)
    m = np.mean(np.stack(valid, axis=0), axis=0)
    n = float(np.linalg.norm(m))
    return m / n if n > 1e-12 else np.zeros(512, dtype=np.float64)


def entity_max_cosine(anchor: np.ndarray, cand: np.ndarray) -> float:
    """Max pairwise cosine between anchor tag rows (n_a, d) and candidate tag rows (n_c, d)."""
    if anchor.size == 0 or cand.size == 0:
        return 0.0
    anchor = np.asarray(anchor, dtype=np.float64).reshape(-1, anchor.shape[-1])
    cand = np.asarray(cand, dtype=np.float64).reshape(-1, cand.shape[-1])
    an = np.linalg.norm(anchor, axis=1, keepdims=True)
    cn = np.linalg.norm(cand, axis=1, keepdims=True)
    if not ((an > 1e-12).any() and (cn > 1e-12).any()):
        return 0.0
    sims = (anchor @ cand.T) / (an @ cn.T + 1e-12)
    return float(np.max(sims))


def anchor_app_ids(*, query_app_id: int, history_app_ids: set[int], mode: AnchorMode) -> list[int]:
    """Source app ids for building an anchor (query-only or history union)."""
    if mode == "query" or not history_app_ids:
        return [int(query_app_id)]
    return sorted(int(a) for a in history_app_ids)


def field_similarity(
    *,
    field: str,
    anchor_apps: list[int],
    cand_app_id: int,
    sim_mode: SimMode,
) -> float:
    """Cosine similarity for one taxonomy field between anchor context and a candidate game."""
    if sim_mode == "pooled":
        anchor_vecs = [pooled_by_app.get(a, {}).get(field, np.zeros(512)) for a in anchor_apps]
        anchor = mean_l2_normalize(anchor_vecs) if len(anchor_vecs) > 1 else parse_pooled_use(anchor_vecs[0])
        cand = parse_pooled_use(pooled_by_app.get(int(cand_app_id), {}).get(field))
        return cosine_sim(anchor, cand)
    anchor_parts = [entity_by_app.get(a, {}).get(field, np.zeros((0, 512))) for a in anchor_apps]
    anchor = np.concatenate(anchor_parts, axis=0) if anchor_parts else np.zeros((0, 512))
    cand = entity_by_app.get(int(cand_app_id), {}).get(field, np.zeros((0, 512)))
    return entity_max_cosine(anchor, cand)


def metadata_pool_scores(
    pool_app_ids: list[int],
    retrieval_scores: list[float] | np.ndarray,
    *,
    fields: tuple[str, ...],
    alpha: float,
    sim_mode: SimMode,
    query_app_id: int,
    history_app_ids: set[int],
    anchor_mode: AnchorMode,
) -> np.ndarray:
    """α × norm(retr) + (1−α) × norm(weighted mean field cosine). α=1 → retrieval only."""
    apps = anchor_app_ids(query_app_id=query_app_id, history_app_ids=history_app_ids, mode=anchor_mode)
    meta = np.empty(len(pool_app_ids), dtype=np.float64)
    for i, app_id in enumerate(pool_app_ids):
        meta[i] = float(np.mean([field_similarity(field=f, anchor_apps=apps, cand_app_id=int(app_id), sim_mode=sim_mode) for f in fields]))
    retr = minmax_norm(np.asarray(retrieval_scores, dtype=np.float64))
    meta_n = minmax_norm(meta)
    return alpha * retr + (1.0 - alpha) * meta_n

def raw_metadata_pool_scores(
    pool_app_ids: list[int],
    *,
    fields: tuple[str, ...],
    sim_mode: SimMode,
    query_app_id: int,
    history_app_ids: set[int],
    anchor_mode: AnchorMode,
) -> np.ndarray:
    """Pure taxonomy USE metadata signal (no retrieval blend) for D1+meta ablation."""
    apps = anchor_app_ids(query_app_id=query_app_id, history_app_ids=history_app_ids, mode=anchor_mode)
    meta = np.empty(len(pool_app_ids), dtype=np.float64)
    for i, app_id in enumerate(pool_app_ids):
        meta[i] = float(
            np.mean(
                [
                    field_similarity(field=f, anchor_apps=apps, cand_app_id=int(app_id), sim_mode=sim_mode)
                    for f in fields
                ]
            )
        )
    return meta


def d1_plus_metadata_pool_scores(
    pool_app_ids: list[int],
    retrieval_scores: list[float] | np.ndarray,
    *,
    w_meta: float,
    fields: tuple[str, ...],
    sim_mode: SimMode,
    query_app_id: int,
    history_app_ids: set[int],
    anchor_mode: AnchorMode,
) -> np.ndarray:
    """Convex mix: ``(1 - w_meta) * norm(D1) + w_meta * norm(metadata USE cosine)``."""
    d1 = score_logpop_blend(
        pool_app_ids,
        retrieval_scores,
        alpha=D1_ALPHA,
        pop_row=pop_row,
        app_to_row=app_to_row,
    )
    meta = raw_metadata_pool_scores(
        pool_app_ids,
        fields=fields,
        sim_mode=sim_mode,
        query_app_id=query_app_id,
        history_app_ids=history_app_ids,
        anchor_mode=anchor_mode,
    )
    if w_meta <= 0.0:
        return d1
    if w_meta >= 1.0:
        return minmax_norm(meta)
    return (1.0 - w_meta) * minmax_norm(d1) + w_meta * minmax_norm(meta)



def pool_scores_to_ranked_indices(pool_app_ids: list[int], pool_scores: np.ndarray, *, k_final: int) -> np.ndarray:
    """Map pool scores → top-k catalog row indices."""
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


def stratified_ex_idx_split(pools: list[dict[str, Any]], *, tune_frac: float, seed: int) -> tuple[set[int], set[int]]:
    """Split ex_idx into fit/tune stratified by slice_name."""
    rng = np.random.default_rng(seed)
    by_slice: dict[str, list[int]] = {}
    for row in pools:
        by_slice.setdefault(str(row["slice_name"]), []).append(int(row["ex_idx"]))
    fit_ids: set[int] = set()
    tune_ids: set[int] = set()
    for ids in by_slice.values():
        ids_arr = np.asarray(sorted(ids))
        rng.shuffle(ids_arr)
        n_tune = max(1, int(round(len(ids_arr) * tune_frac)))
        tune_ids.update(int(x) for x in ids_arr[:n_tune])
        fit_ids.update(int(x) for x in ids_arr[n_tune:])
    return fit_ids, tune_ids


_, tune_ex_idx = stratified_ex_idx_split(train_pools, tune_frac=TUNE_FRAC, seed=SPLIT_SEED)
train_tune = [r for r in train_pools if int(r["ex_idx"]) in tune_ex_idx]
print(f"train_tune={len(train_tune):,} / {len(train_pools):,}")


train_tune=5,169 / 51,691


## Core Analysis

In [5]:
def mean_ndcg_slice_a(
    pools: list[dict[str, Any]],
    *,
    fields: tuple[str, ...],
    alpha: float,
    sim_mode: SimMode,
    anchor_mode: AnchorMode = "query",
) -> float:
    """Mean Slice A NDCG@K for train_tune hyperparameter search."""
    vals: list[float] = []
    for row in pools:
        if int(row["n_eval_targets"]) < 2:
            continue
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = json.loads(row["retrieved_scores_json"])
        ex_idx = int(row["ex_idx"])
        history = history_apps_by_ex.get(ex_idx, set()) if anchor_mode == "history" else set()
        blend = metadata_pool_scores(
            pool_apps, ret_sc, fields=fields, alpha=alpha, sim_mode=sim_mode,
            query_app_id=int(row["query_app_id"]), history_app_ids=history, anchor_mode=anchor_mode,
        )
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
        vals.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(vals)) if vals else float("nan")


grid_rows: list[dict[str, Any]] = []
for sim_mode in SIM_MODES:
    for preset_name, fields in FIELD_PRESETS.items():
        for alpha in BLEND_ALPHAS:
            grid_rows.append({
                "sim_mode": sim_mode,
                "field_preset": preset_name,
                "fields": ",".join(fields),
                "alpha": alpha,
                "train_tune_NDCG_slice_a_query": mean_ndcg_slice_a(
                    train_tune, fields=fields, alpha=alpha, sim_mode=sim_mode, anchor_mode="query",
                ),
            })

train_grid = pd.DataFrame(grid_rows).sort_values("train_tune_NDCG_slice_a_query", ascending=False)
best = train_grid.iloc[0]
BEST_SIM_MODE = str(best["sim_mode"])
BEST_FIELDS = tuple(FIELD_PRESETS[str(best["field_preset"])])
BEST_ALPHA = float(best["alpha"])
BEST_PRESET = str(best["field_preset"])

display(Markdown("### Train_tune grid (query anchor · Slice A NDCG@10)"))
display(train_grid.head(15))
print(f"BEST: sim_mode={BEST_SIM_MODE} preset={BEST_PRESET} alpha={BEST_ALPHA} fields={BEST_FIELDS}")
train_grid.to_csv(SPIKE_OUT / "v2a_embed_train_tune_grid.csv", index=False)


### Train_tune grid (query anchor · Slice A NDCG@10)

,sim_mode,field_preset,fields,alpha,train_tune_NDCG_slice_a_query
18,pooled,genre_theme_kw,"genres,themes,keywords",0.0,0.089080
0,pooled,all5,"genres,themes,keywords,game_modes,player_persp...",0.0,0.085621
19,pooled,genre_theme_kw,"genres,themes,keywords",0.2,0.085276
1,pooled,all5,"genres,themes,keywords,game_modes,player_persp...",0.2,0.081424
24,entity_max,all5,"genres,themes,keywords,game_modes,player_persp...",0.0,0.079558
42,entity_max,genre_theme_kw,"genres,themes,keywords",0.0,0.079215
25,entity_max,all5,"genres,themes,keywords,game_modes,player_persp...",0.2,0.072073
6,pooled,no_keywords,"genres,themes,game_modes,player_perspectives",0.0,0.068803
12,pooled,genre_theme,"genres,themes",0.0,0.067077
43,entity_max,genre_theme_kw,"genres,themes,keywords",0.2,0.065251


BEST: sim_mode=pooled preset=genre_theme_kw alpha=0.0 fields=('genres', 'themes', 'keywords')


## D1 + metadata ablation (`_logpop_blend`)

Follow-up to the retr+meta grid: blend **shipped D1** with **pure taxonomy USE metadata** (fixed `BEST_SIM_MODE`, `BEST_FIELDS`). Tune `w_meta` on train_tune (query anchor); val reports `*_logpop_blend` methods alongside baselines.


In [6]:
def mean_ndcg_slice_a_plus_d1(
    pools: list[dict[str, Any]],
    *,
    fields: tuple[str, ...],
    w_meta: float,
    anchor_mode: AnchorMode = "query",
) -> float:
    """Mean Slice A NDCG@K for D1+metadata blend — train_tune search only."""
    vals: list[float] = []
    for row in pools:
        if int(row["n_eval_targets"]) < 2:
            continue
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = json.loads(row["retrieved_scores_json"])
        ex_idx = int(row["ex_idx"])
        history = history_apps_by_ex.get(ex_idx, set()) if anchor_mode == "history" else set()
        blend = d1_plus_metadata_pool_scores(
            pool_apps,
            ret_sc,
            w_meta=w_meta,
            fields=fields,
            sim_mode=BEST_SIM_MODE,  # type: ignore[arg-type]
            query_app_id=int(row["query_app_id"]),
            history_app_ids=history,
            anchor_mode=anchor_mode,
        )
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
        vals.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(vals)) if vals else float("nan")


plus_d1_grid_rows: list[dict[str, Any]] = []
for w_meta in PLUS_D1_WEIGHTS:
    plus_d1_grid_rows.append(
        {
            "w_meta": w_meta,
            "train_tune_NDCG_slice_a_query": mean_ndcg_slice_a_plus_d1(
                train_tune, fields=BEST_FIELDS, w_meta=w_meta, anchor_mode="query"
            ),
        }
    )

plus_d1_train_grid = pd.DataFrame(plus_d1_grid_rows).sort_values(
    "train_tune_NDCG_slice_a_query", ascending=False
)
plus_d1_best = plus_d1_train_grid.iloc[0]
BEST_W_META = float(plus_d1_best["w_meta"])

display(Markdown("### Train_tune grid — D1 + metadata (query anchor · Slice A NDCG@10)"))
display(plus_d1_train_grid)
print(f"BEST_PLUS_D1: w_meta={BEST_W_META} fields={BEST_FIELDS}")

plus_d1_train_grid.to_csv(SPIKE_OUT / "v2a_embed_plus_d1_train_tune_grid.csv", index=False)


### Train_tune grid — D1 + metadata (query anchor · Slice A NDCG@10)

,w_meta,train_tune_NDCG_slice_a_query
2,0.10,0.171055
1,0.05,0.170353
3,0.15,0.170155
0,0.00,0.170115
4,0.20,0.168124
5,0.30,0.162986


BEST_PLUS_D1: w_meta=0.1 fields=('genres', 'themes', 'keywords')


## Evaluation

In [7]:
def per_example_ranking_row(
    row: dict[str, Any],
    *,
    method: str,
    score_fn: Callable[..., np.ndarray] | None = None,
    score_kwargs: dict[str, Any] | None = None,
    oracle: bool = False,
    catalog_pop: bool = False,
) -> dict[str, Any] | None:
    """Ranking metrics for one val example; includes oracle columns for aggregation contract."""
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    if not positives:
        return None
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)
    oracle_indices = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)
    if oracle:
        ranked = oracle_indices[:K_FINAL]
    elif catalog_pop:
        s = np.asarray(pop_row, dtype=np.float64).copy()
        qrow = app_to_row.get(int(row["query_app_id"]))
        if qrow is not None:
            s[qrow] = -np.inf
        ranked = _rank_rows(s)[:K_FINAL]
    elif score_fn is None:
        ranked = pool_scores_to_ranked_indices(pool_apps, np.asarray(ret_sc), k_final=K_FINAL)
    else:
        blend = score_fn(pool_apps, ret_sc, **(score_kwargs or {}))
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
    return {
        "method": method,
        "ex_idx": int(row["ex_idx"]),
        "slice_name": row["slice_name"],
        "n_eval_targets": int(row["n_eval_targets"]),
        "n_support_train": int(row.get("n_support_train", 0)),
        "query_app_id": int(row["query_app_id"]),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "Precision@K": precision_at_k(ranked, positives, K_FINAL, app_ids),
        "Recall@K": recall_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
        "OracleHit@K": hit_rate_at_k(oracle_indices, positives, K_FINAL, app_ids),
        "OracleNDCG@K": ndcg_at_k(oracle_indices, positives, K_FINAL, app_ids),
    }


def make_embed_score_fn(anchor_mode: AnchorMode, ex_idx: int, query_app_id: int) -> Callable[..., np.ndarray]:
    def _score(pool_apps: list[int], ret_sc: list[float]) -> np.ndarray:
        return metadata_pool_scores(
            pool_apps, ret_sc, fields=BEST_FIELDS, alpha=BEST_ALPHA, sim_mode=BEST_SIM_MODE,  # type: ignore[arg-type]
            query_app_id=query_app_id,
            history_app_ids=history_apps_by_ex.get(ex_idx, set()),
            anchor_mode=anchor_mode,
        )
    return _score


def make_embed_plus_d1_score_fn(anchor_mode: AnchorMode, ex_idx: int, query_app_id: int) -> Callable[..., np.ndarray]:
    def _score(pool_apps: list[int], ret_sc: list[float]) -> np.ndarray:
        return d1_plus_metadata_pool_scores(
            pool_apps, ret_sc, w_meta=BEST_W_META, fields=BEST_FIELDS, sim_mode=BEST_SIM_MODE,  # type: ignore[arg-type]
            query_app_id=query_app_id,
            history_app_ids=history_apps_by_ex.get(ex_idx, set()),
            anchor_mode=anchor_mode,
        )
    return _score


val_metric_rows: list[dict[str, Any]] = []
for row in val_pools:
    ex_idx = int(row["ex_idx"])
    qid = int(row["query_app_id"])
    for method, kwargs in (
        (POOL_METHOD, {}),
        ("two_tower_v1_heuristic_logpop_blend", {
            "score_fn": score_logpop_blend,
            "score_kwargs": {"alpha": D1_ALPHA, "pop_row": pop_row, "app_to_row": app_to_row},
        }),
        ("popularity_train", {"catalog_pop": True}),
        (f"{POOL_METHOD}_oracle", {"oracle": True}),
        ("two_tower_v1_v2a_embed_query", {"score_fn": make_embed_score_fn("query", ex_idx, qid)}),
        ("two_tower_v1_v2a_embed_history", {"score_fn": make_embed_score_fn("history", ex_idx, qid)}),
        ("two_tower_v1_v2a_embed_query_logpop_blend", {"score_fn": make_embed_plus_d1_score_fn("query", ex_idx, qid)}),
        ("two_tower_v1_v2a_embed_history_logpop_blend", {"score_fn": make_embed_plus_d1_score_fn("history", ex_idx, qid)}),
    ):
        mrow = per_example_ranking_row(row, method=method, **kwargs)
        if mrow:
            val_metric_rows.append(mrow)

df_val = pd.DataFrame(val_metric_rows)
df_val.to_parquet(SPIKE_OUT / "v2a_embed_val_per_example.parquet", index=False)

overall = _table_overall_ranking(df_val)
by_slice = _table_by_slice_for_metrics(df_val, metric_cols=RANKING_REPORT_METRIC_COLS, ranking=True)
by_support = _table_by_support_for_metrics(df_val, metric_cols=RANKING_REPORT_METRIC_COLS, ranking=True)
pop_table, pop_delta, _ = _table_popularity(
    df_ex_metrics=df_val, examples=examples_for_pers, app_ids=app_ids, pop_row=pop_row,
    enable_popularity_decile_diagnostics=True, metric_cols=RANKING_REPORT_METRIC_COLS,
)

overall.to_csv(SPIKE_OUT / "v2a_embed_val_overall.csv", index=False)
by_slice.to_csv(SPIKE_OUT / "v2a_embed_val_by_slice.csv", index=False)
by_support.to_csv(SPIKE_OUT / "v2a_embed_val_by_support.csv", index=False)
pop_table.to_csv(SPIKE_OUT / "v2a_embed_val_by_pop_decile.csv", index=False)
pop_delta.to_csv(SPIKE_OUT / "v2a_embed_val_pop_delta.csv", index=False)

display(Markdown("### Val overall"))
display(overall.sort_values("NDCG@K", ascending=False))
display(Markdown("### Val by slice"))
display(by_slice.sort_values(["slice_name", "NDCG@K"], ascending=[True, False]))


### Val overall

,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K
0,two_tower_v1_oracle,0.51224,0.054144,0.494053,0.494053,0.498310,0.512240,0.51224,0.49831
1,two_tower_v1_v2a_embed_query_logpop_blend,0.19584,0.019816,0.186324,0.066797,0.095345,0.069567,0.51224,0.49831
2,two_tower_v1_v2a_embed_history_logpop_blend,0.19520,0.019744,0.185925,0.064720,0.093633,0.067484,0.51224,0.49831
3,two_tower_v1_heuristic_logpop_blend,0.19328,0.019560,0.184095,0.064321,0.092892,0.067059,0.51224,0.49831
4,popularity_train,0.15112,0.015184,0.146796,0.050594,0.073109,0.052221,0.51224,0.49831
5,two_tower_v1_v2a_embed_query,0.10696,0.010792,0.100016,0.037230,0.052477,0.039852,0.51224,0.49831
6,two_tower_v1_v2a_embed_history,0.10400,0.010480,0.098875,0.030180,0.046493,0.031725,0.51224,0.49831
7,two_tower_v1,0.04680,0.004728,0.043740,0.010325,0.018161,0.011008,0.51224,0.49831


### Val by slice

,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K
0,slice_a_multi_target,two_tower_v1_oracle,0.773793,0.127724,0.460228,0.460228,0.533618,0.773793,0.773793,0.533618
1,slice_a_multi_target,two_tower_v1_v2a_embed_query_logpop_blend,0.281379,0.032138,0.117319,0.035030,0.070334,0.082787,0.773793,0.533618
2,slice_a_multi_target,two_tower_v1_heuristic_logpop_blend,0.271724,0.031172,0.113367,0.034181,0.068322,0.081396,0.773793,0.533618
3,slice_a_multi_target,two_tower_v1_v2a_embed_history_logpop_blend,0.273103,0.031172,0.113195,0.033843,0.068113,0.081498,0.773793,0.533618
4,slice_a_multi_target,two_tower_v1_v2a_embed_query,0.201379,0.021793,0.081660,0.030763,0.055602,0.075979,0.773793,0.533618
5,slice_a_multi_target,popularity_train,0.128276,0.014069,0.053728,0.019418,0.035444,0.047469,0.773793,0.533618
6,slice_a_multi_target,two_tower_v1_v2a_embed_history,0.146207,0.016000,0.057846,0.017229,0.035130,0.043865,0.773793,0.533618
7,slice_a_multi_target,two_tower_v1,0.091034,0.009931,0.038280,0.009408,0.020537,0.021192,0.773793,0.533618
8,slice_b_single_target,two_tower_v1_oracle,0.496136,0.049614,0.496136,0.496136,0.496136,0.496136,0.496136,0.496136
9,slice_b_single_target,two_tower_v1_v2a_embed_query_logpop_blend,0.190573,0.019057,0.190573,0.068753,0.096885,0.068753,0.496136,0.496136


## Personalization

In [8]:
def popularity_train_score(ex: dict[str, Any]) -> np.ndarray:
    s = np.asarray(pop_row, dtype=np.float64).copy()
    row = app_to_row.get(int(ex["query_app_id"]))
    if row is not None:
        s[row] = -np.inf
    return s.astype(np.float32)


def frozen_pool_score(ex: dict[str, Any]) -> np.ndarray:
    row = val_pools_by_ex[int(ex["ex_idx"])]
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_apps, ret_sc):
        full[int(app_to_row[int(app_id)])] = score
    return full.astype(np.float32)


def make_pers_score_fn(anchor_mode: AnchorMode) -> Callable[[dict[str, Any]], np.ndarray]:
    def score(ex: dict[str, Any]) -> np.ndarray:
        row = val_pools_by_ex[int(ex["ex_idx"])]
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        blend = metadata_pool_scores(
            pool_apps, ret_sc, fields=BEST_FIELDS, alpha=BEST_ALPHA, sim_mode=BEST_SIM_MODE,  # type: ignore[arg-type]
            query_app_id=int(ex["query_app_id"]),
            history_app_ids=history_apps_by_ex.get(int(ex["ex_idx"]), set()),
            anchor_mode=anchor_mode,
        )
        full = np.full(len(app_ids), -np.inf, dtype=np.float64)
        for app_id, sc in zip(pool_apps, blend):
            full[int(app_to_row[int(app_id)])] = float(sc)
        return full.astype(np.float32)
    return score


def make_pers_plus_d1_score_fn(anchor_mode: AnchorMode) -> Callable[[dict[str, Any]], np.ndarray]:
    def score(ex: dict[str, Any]) -> np.ndarray:
        row = val_pools_by_ex[int(ex["ex_idx"])]
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        blend = d1_plus_metadata_pool_scores(
            pool_apps, ret_sc, w_meta=BEST_W_META, fields=BEST_FIELDS, sim_mode=BEST_SIM_MODE,  # type: ignore[arg-type]
            query_app_id=int(ex["query_app_id"]),
            history_app_ids=history_apps_by_ex.get(int(ex["ex_idx"]), set()),
            anchor_mode=anchor_mode,
        )
        full = np.full(len(app_ids), -np.inf, dtype=np.float64)
        for app_id, sc in zip(pool_apps, blend):
            full[int(app_to_row[int(app_id)])] = float(sc)
        return full.astype(np.float32)
    return score


def d1_frozen_score(ex: dict[str, Any]) -> np.ndarray:
    row = val_pools_by_ex[int(ex["ex_idx"])]
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    blend = score_logpop_blend(pool_apps, ret_sc, alpha=D1_ALPHA, pop_row=pop_row, app_to_row=app_to_row)
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, sc in zip(pool_apps, blend):
        full[int(app_to_row[int(app_id)])] = float(sc)
    return full.astype(np.float32)


pers_methods = {
    "popularity_train": popularity_train_score,
    POOL_METHOD: frozen_pool_score,
    "two_tower_v1_heuristic_logpop_blend": d1_frozen_score,
    "two_tower_v1_v2a_embed_query": make_pers_score_fn("query"),
    "two_tower_v1_v2a_embed_history": make_pers_score_fn("history"),
    "two_tower_v1_v2a_embed_query_logpop_blend": make_pers_plus_d1_score_fn("query"),
    "two_tower_v1_v2a_embed_history_logpop_blend": make_pers_plus_d1_score_fn("history"),
}

personalization = _table_personalization(
    methods=pers_methods, examples=examples_for_pers, X=X_emb, app_ids=app_ids,
    pop_row=pop_row, k_personalization=K_PERSONALIZATION,
)
overall_pers = _append_personalization_metrics(overall.copy(), personalization, on_keys=["method"])
personalization.to_csv(SPIKE_OUT / "v2a_embed_val_personalization.csv", index=False)
overall_pers.to_csv(SPIKE_OUT / "v2a_embed_val_overall_with_personalization.csv", index=False)

display(Markdown("### Personalization"))
display(personalization.sort_values("PersonalizationGapVsPopularity@10", ascending=False))


### Personalization

,method,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
1,two_tower_v1,0.261639,0.987302,12.167556,0.995623
5,two_tower_v1_v2a_embed_query,0.187939,0.977778,8.660892,0.952469
3,two_tower_v1_v2a_embed_history,0.189916,0.977778,8.430697,0.938952
6,two_tower_v1_v2a_embed_query_logpop_blend,0.202787,0.511111,6.296105,0.726256
4,two_tower_v1_v2a_embed_history_logpop_blend,0.203100,0.495238,6.281845,0.724458
2,two_tower_v1_heuristic_logpop_blend,0.204632,0.501587,6.272161,0.720097
0,popularity_train,0.212335,0.034921,5.317471,0.000000


# Key Findings

**Finding 1:**  
Best retr+meta train_tune: pooled, `genre_theme_kw`, **α=0**, Slice A **0.089** — val retr+meta alone still below D1.

**Finding 2 (retr+meta — kill):**  
`two_tower_v1_v2a_embed_query`: val **0.052** / **0.035** overall / Slice A vs D1 **0.093** / **0.068**.

**Finding 3 (`_embed_query_logpop_blend` — promotion candidate):**  
`two_tower_v1_v2a_embed_query_logpop_blend` (`w_meta=0.1`, pooled, `genre_theme_kw`): val **0.095** / **0.070** overall / Slice A — **beats D1**; personalization gap **0.726** vs D1 **0.720**. Slightly ahead of 019 Jaccard logpop_blend on Slice A (**0.070** vs **0.070** rounded; **0.070334** vs **0.069860**).

**Finding 4:**  
**pooled** beat **entity_max** on train_tune. History logpop_blend **0.094** / **0.068** — below query on Slice A.

**Unexpected Results:**  
Semantic embed alone did not beat D1; D1+embed at small `w_meta` does — same pattern as 019 Jaccard logpop_blend.


# Recommendation / Next Steps

**Recommended Action:**  
**Kill** retr+meta-only embed. **Promotion candidate:** `two_tower_v1_v2a_embed_query_logpop_blend` — passes val promotion bar vs D1. Compare vs `recs_019` `two_tower_v1_v2a_query_metadata_logpop_blend`; wire winner into eval job + `ranking_decision_log.md`.

**Risks:**  
Small catalog; two near-tie metadata mechanisms — pick one for ship to avoid complexity.

**Follow-up:**  
Head-to-head 019 vs 020 logpop_blend; V2b summary spike with logpop_blend pattern if embed path wins.
